### 01_explore.ipynb

In [10]:
import sys
print(sys.executable)

/Users/Anneleonore/Documents/githubportfolio/delivery-agent/.venv/bin/python


In [11]:
import pandas as pd

df = pd.read_csv("../data/historical_data.csv")
print(df.shape)
df.head()

(197428, 16)


,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


In [12]:
# --- Prepare: timestamps and delivery time in minutes ---
df["created_at"] = pd.to_datetime(df["created_at"])
df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"])
df["delivery_min"] = (df["actual_delivery_time"] - df["created_at"]).dt.total_seconds() / 60

def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

section("1. Period covered")
print(f"First order: {df['created_at'].min()}")
print(f"Last order:  {df['created_at'].max()}")
print(f"Orders per month:\n{df['created_at'].dt.to_period('M').value_counts().sort_index()}")

section("2. Missing values (%)")
missing = df.isna().mean().mul(100).round(1)
print(missing[missing > 0].sort_values(ascending=False))

section("3. Delivery time (minutes)")
print(df["delivery_min"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))
print(f"\nLonger than 2 hours: {(df['delivery_min'] > 120).sum():,} orders")
print(f"Longer than 1 day:   {(df['delivery_min'] > 1440).sum():,} orders")

section("4. Markets and store categories")
print(df["market_id"].value_counts(dropna=False).sort_index())
print(f"\nNumber of store categories: {df['store_primary_category'].nunique()}")
print(f"Top 10:\n{df['store_primary_category'].value_counts().head(10)}")

section("5. Courier data: odd values")
print(f"Negative on-shift couriers:     {(df['total_onshift_dashers'] < 0).sum():,}")
print(f"Negative busy couriers:         {(df['total_busy_dashers'] < 0).sum():,}")
print(f"Negative outstanding orders:    {(df['total_outstanding_orders'] < 0).sum():,}")
print(f"More busy than on shift:        {(df['total_busy_dashers'] > df['total_onshift_dashers']).sum():,}")

section("6. Order value (probably in cents)")
print(df[["subtotal", "min_item_price", "max_item_price", "total_items"]].describe().round(0))
print(f"\nNegative item prices: {(df['min_item_price'] < 0).sum():,}")


1. Period covered
First order: 2014-10-19 05:24:15
Last order:  2015-02-18 06:00:44
Orders per month:
created_at
2014-10         1
2015-01     68482
2015-02    128945
Freq: M, Name: count, dtype: int64

2. Missing values (%)
total_onshift_dashers                           8.2
total_busy_dashers                              8.2
total_outstanding_orders                        8.2
store_primary_category                          2.4
market_id                                       0.5
order_protocol                                  0.5
estimated_store_to_consumer_driving_duration    0.3
dtype: float64

3. Delivery time (minutes)
count    197421.0
mean         48.5
std         320.5
min           1.7
5%           24.9
25%          35.1
50%          44.3
75%          56.4
95%          81.2
99%         107.9
max      141947.6
Name: delivery_min, dtype: float64

Longer than 2 hours: 1,090 orders
Longer than 1 day:   3 orders

4. Markets and store categories
market_id
1.0    38037
2.0    55058


### Drops orders that are unusable or clearly wrong
- Orders without a delivery time
- The single order from October 2014
- Orders without a market

### Flags, but keeps, unusually long deliveries
- Deliveries recorded as taking more than five hours are marked in a separate column, so they can be left out of averages and the model where needed.

### Repairs impossible values

- Negative numbers of couriers or outstanding orders are treated as missing.

### Fills in missing store categories

- A missing category is taken from the same store's other orders; otherwise it becomes "unknown".

### Makes the data more readable

- Amounts are converted from cents to dollars, and market numbers are stored as whole numbers.

In [16]:
import numpy as np

clean = df.copy()
log = []

def drop(mask, reason):
    """Remove rows where mask is True and record how many."""
    global clean
    log.append((reason, int(mask.sum())))
    clean = clean[~mask]

drop(clean["delivery_min"].isna(), "No delivery time")
drop(clean["created_at"] < "2015-01-01", "Stray order from 2014")
drop(clean["market_id"].isna(), "No market")

# Flag (but keep) very long deliveries: over 5 hours, possibly a data error
clean["long_delivery"] = clean["delivery_min"] > 300

# Impossible courier values -> missing
for col in ["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"]:
    clean.loc[clean[col] < 0, col] = np.nan

# Missing store category: take the store's most common category, otherwise "unknown"
store_cat = (
    clean.dropna(subset=["store_primary_category"])
    .groupby("store_id")["store_primary_category"]
    .agg(lambda s: s.mode().iloc[0])
)
clean["store_primary_category"] = (
    clean["store_primary_category"]
    .fillna(clean["store_id"].map(store_cat))
    .fillna("unknown")
)

# Cents -> dollars
for col in ["subtotal", "min_item_price", "max_item_price"]:
    clean[col] = clean[col] / 100

clean["market_id"] = clean["market_id"].astype(int)

# Timestamps are in UTC; the region's local time is US Pacific (PST, UTC-8, no daylight saving in Jan-Feb)
clean["created_local"] = clean["created_at"] - pd.Timedelta(hours=8)

print(f"Start:   {len(df):,} orders")
for reason, n in log:
    print(f"  − {n:>6,}  {reason}")
print(f"Clean:   {len(clean):,} orders ({len(clean) / len(df):.1%} kept)")
print(f"Store category still unknown: {(clean['store_primary_category'] == 'unknown').sum():,}")
print(f"Flagged as long (over 5 hours): {clean['long_delivery'].sum():,}")


Start:   197,428 orders
  −      7  No delivery time
  −      1  Stray order from 2014
  −    987  No market
Clean:   196,433 orders (99.5% kept)
Store category still unknown: 866
Flagged as long (over 5 hours): 34


In [19]:
# Save the clean data for the next notebooks
clean.to_parquet("../data/deliveries_clean.parquet", index=False)

In [20]:
typical = clean[~clean["long_delivery"]].copy()

# Courier pressure: how busy the market is when the order is placed
typical["busy_share"] = (typical["total_busy_dashers"] / typical["total_onshift_dashers"]).clip(upper=1.5)
typical["orders_per_dasher"] = typical["total_outstanding_orders"] / typical["total_onshift_dashers"]
typical["hour"] = typical["created_local"].dt.hour
typical["weekday"] = typical["created_local"].dt.day_name()

def summary(by, data=typical):
    """Median delivery time and number of orders per group."""
    return (data.groupby(by, observed=True)["delivery_min"]
                .agg(median_min="median", orders="size")
                .round(1))

section("A. By hour of the day (US Pacific)")
print(summary("hour"))

section("B. By weekday")
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(summary("weekday").reindex(order))

section("C. By courier pressure: outstanding orders per courier on shift")
bins = [0, 0.5, 1, 1.5, 2, np.inf]
labels = ["< 0.5", "0.5–1", "1–1.5", "1.5–2", "2+"]
typical["pressure"] = pd.cut(typical["orders_per_dasher"], bins=bins, labels=labels)
print(summary("pressure"))

section("D. By market")
print(summary("market_id"))

section("E. By store category (10 most common)")
top10 = typical["store_primary_category"].value_counts().head(10).index
print(summary("store_primary_category", typical[typical["store_primary_category"].isin(top10)])
      .sort_values("median_min"))

section("F. By order size (number of items)")
typical["items_group"] = pd.cut(typical["total_items"], bins=[0, 1, 2, 4, 8, np.inf],
                                labels=["1", "2", "3–4", "5–8", "9+"])
print(summary("items_group"))


A. By hour of the day (US Pacific)
      median_min  orders
hour                    
6           60.8      40
7           47.5     535
8           44.0    2102
9           40.9    3388
10          37.7    5069
11          43.2   13464
12          41.3   15485
13          38.2   11412
14          38.9    8779
15          39.9    8125
16          40.9   12606
17          45.9   28023
18          51.2   36749
19          49.1   26959
20          41.0   15189
21          36.7    7057
22          37.9    1406
23          48.6      11

B. By weekday
           median_min  orders
weekday                      
Monday           44.2   24426
Tuesday          41.6   23800
Wednesday        43.7   25148
Thursday         41.8   26927
Friday           48.0   32946
Saturday         45.3   32681
Sunday           44.8   30471

C. By courier pressure: outstanding orders per courier on shift
          median_min  orders
pressure                    
< 0.5           36.8    6493
0.5–1           38.4   4998